In [25]:
# put this at the top of every notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"
#pio.renderers.default = "browser"   # always opens in browser, no nbformat needed


In [26]:
# Cell 1 — setup
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

client = StockClient()
quant  = QuantAnalytics(client)
opt    = OptionsAnalyzer(client, "SPY")

In [27]:
# Cell 2 — expiries
print(opt.expiries())


['2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-15', '2026-05-22', '2026-05-29', '2026-06-05', '2026-06-18', '2026-06-30', '2026-07-17', '2026-07-31', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15']


In [28]:
# Cell 3 — front month chain
expiry = opt.nearest_expiry(0)
print(f"Front month: {expiry}")
df = opt.chain(expiry)
print(df.head(20))

Front month: 2026-05-04
        expiry type  strike  bid   ask  last_price  volume  open_interest  \
0   2026-05-04  put   500.0  0.0  0.01        0.01    13.0            967   
1   2026-05-04  put   505.0  0.0  0.01        0.01     0.0            105   
2   2026-05-04  put   510.0  0.0  0.01        0.02     0.0            204   
3   2026-05-04  put   515.0  0.0  0.01        0.01     2.0             17   
4   2026-05-04  put   520.0  0.0  0.01        0.02     0.0            100   
5   2026-05-04  put   525.0  0.0  0.01        0.01    51.0             61   
6   2026-05-04  put   530.0  0.0  0.01        0.01  1030.0           1036   
7   2026-05-04  put   540.0  0.0  0.01        0.03     0.0             34   
8   2026-05-04  put   545.0  0.0  0.01        0.05     0.0             49   
9   2026-05-04  put   550.0  0.0  0.01        0.04     0.0             30   
10  2026-05-04  put   555.0  0.0  0.01        0.03     0.0            111   
11  2026-05-04  put   560.0  0.0  0.01        0.02  

In [29]:
# Cell 4 — summary
print(opt.summary(expiry))

{'symbol': 'SPY', 'expiry': '2026-05-04', 'days_to_expiry': 2, 'n_strikes': 124, 'total_call_oi': 86824, 'total_put_oi': 151945, 'total_call_vol': 586717, 'total_put_vol': 687643, 'put_call_ratio_oi': 1.75, 'put_call_ratio_vol': 1.172, 'max_pain_strike': 714.0, 'max_oi_call_strike': np.float64(725.0), 'max_oi_put_strike': np.float64(710.0), 'avg_call_iv': 0.2564, 'avg_put_iv': 0.3589, 'iv_skew': 0.1025}


In [30]:
# Cell 5 — max pain (knockout price)
print(f"Max pain: ${opt.max_pain(expiry):,.2f}")

Max pain: $714.00


In [31]:
# Cell 6 — put/call ratio
print(opt.put_call_ratio())

        expiry  days_to_expiry  call_oi   put_oi  put_call_ratio
0   2026-05-04               2    86824   151945           1.750
1   2026-05-05               3    17345    32205           1.857
2   2026-05-06               4    11723    19338           1.650
3   2026-05-07               5     9166    16726           1.825
4   2026-05-08               6   227236   563203           2.478
5   2026-05-15              13   726062  2805970           3.865
6   2026-05-22              20    92129   533164           5.787
7   2026-05-29              27   229530  1290645           5.623
8   2026-06-05              34    18012    32269           1.792
9   2026-06-18              47   831692  1941389           2.334
10  2026-06-30              59   158802   301518           1.899
11  2026-07-17              76   174294   299620           1.719
12  2026-07-31              90    82616   139049           1.683
13  2026-08-21             111   118291   136053           1.150
14  2026-08-31           

In [32]:
plots.options_chain(opt, opt.nearest_expiry(1)).show()

In [33]:
plots.options_oi_profile(opt, opt.nearest_expiry(1)).show()

In [34]:
plots.options_put_call(opt).show()

In [35]:
plots.options_surface(opt, max_expiries=6).show()


In [36]:
# front 12 expiries (~1 year for weeklies, or full year for monthlies)
plots.options_max_pain(opt, max_expiries=12).show()



In [37]:
# more expiries
plots.options_max_pain(opt, max_expiries=24).show()

In [39]:
plots.options_gex(opt, max_expiries=12).show()


In [41]:
plots.options_unusual(opt, vol_oi_threshold=3.0, min_volume=100).show()

In [42]:
df = opt.greeks(opt.nearest_expiry(0))
print("Greeks columns:", df.columns.tolist())

Greeks columns: ['expiry', 'type', 'strike', 'bid', 'ask', 'last_price', 'volume', 'open_interest', 'implied_volatility', 'in_the_money', 'contract', 'delta', 'gamma', 'theta', 'vega', 'rho', 'theoretical_price', 'greek_source']


In [44]:
#opt = OptionsAnalyzer(client, "SPY")

# Greeks

print(df[["type","strike","delta","gamma","theta","vega","rho","greek_source"]].head(10))


  type  strike  delta     gamma   theta  vega  rho   greek_source
0  put   500.0   -0.0  0.000001 -0.0009   0.0 -0.0  black_scholes
1  put   505.0   -0.0  0.000001 -0.0009   0.0 -0.0  black_scholes
2  put   510.0   -0.0  0.000001 -0.0009   0.0 -0.0  black_scholes
3  put   515.0   -0.0  0.000001 -0.0009   0.0 -0.0  black_scholes
4  put   520.0   -0.0  0.000001 -0.0009   0.0 -0.0  black_scholes
5  put   525.0   -0.0  0.000001 -0.0008   0.0 -0.0  black_scholes
6  put   530.0   -0.0  0.000001 -0.0006   0.0 -0.0  black_scholes
7  put   540.0   -0.0  0.000001 -0.0007   0.0 -0.0  black_scholes
8  put   545.0   -0.0  0.000001 -0.0007   0.0 -0.0  black_scholes
9  put   550.0   -0.0  0.000001 -0.0007   0.0 -0.0  black_scholes


In [45]:
# GEX
strike_gex = opt.gex_by_strike()
by_exp     = opt.gex_by_expiry()
total      = opt.gex_total()

flip     = total.get("flip_strike")
flip_str = f"${flip:,.2f}" if flip is not None else "none detected"
print(f"\nRegime:          {total['regime_label']}")
print(f"Total GEX:       ${total['total_net_gex']/1e6:.1f}M")
print(f"GEX flip:        {flip_str}")
print(f"Dominant expiry: {total.get('dominant_expiry', '—')}")
# Unusual activity
unusual = opt.unusual_activity(vol_oi_threshold=3.0, min_volume=100)
print(f"\nUnusual activity — {len(unusual)} strikes flagged")
if not unusual.empty:
    print(unusual[["type","strike","expiry","volume",
                   "open_interest","vol_oi_ratio","signal"]].head(10))


Regime:          Positive GEX — market makers are net long gamma. They sell rallies and buy dips → price-stabilising, expect lower realised volatility.
Total GEX:       $1681.1M
GEX flip:        $724.00
Dominant expiry: 2026-05-15

Unusual activity — 112 strikes flagged
   type  strike      expiry    volume  open_interest  vol_oi_ratio     signal
0   put   721.0  2026-05-04   74825.0            584        128.12  🚨 Unusual
1   put   722.0  2026-05-04  102303.0           1100         93.00  🚨 Unusual
2   put   723.0  2026-05-04   52700.0            711         74.12  🚨 Unusual
3  call   723.0  2026-05-04  114391.0           2684         42.62  🚨 Unusual
4  call   722.0  2026-05-04   76221.0           2104         36.23  🚨 Unusual
5   put   724.0  2026-05-04   35867.0           1224         29.30  🚨 Unusual
6   put   720.0  2026-05-04   70138.0           2725         25.74  🚨 Unusual
7  call   724.0  2026-05-04   62285.0           2619         23.78  🚨 Unusual
8  call   721.0  2026-05-0

In [46]:
total = opt.gex_total()
print(total.keys())

dict_keys(['symbol', 'total_call_gex', 'total_put_gex', 'total_net_gex', 'regime', 'regime_label', 'flip_strike', 'dominant_expiry'])


In [47]:
by_exp = opt.gex_by_expiry()
print(by_exp.columns.tolist())
print(by_exp.head(2))

['expiry', 'days_to_expiry', 'call_gex', 'put_gex', 'net_gex', 'abs_gex']
       expiry  days_to_expiry      call_gex       put_gex       net_gex  \
0  2026-05-04               2  1.405178e+09 -1.110826e+09  2.943514e+08   
1  2026-05-05               3  1.798711e+08 -1.278825e+08  5.198863e+07   

        abs_gex  
0  1.109117e+09  
1  1.504603e+08  


In [48]:
from yfinance_api3.classes.pricing import (
    ContractType, Space, ExerciseType
)
from yfinance_api3.classes.positions_book import (
    PositionsBook, PortfolioBook, WatchList
)

# ── Build INTC book ───────────────────────────────────────────────────────
book = PositionsBook(
    symbol         = "INTC",
    spot           = 93.22,
    vol            = 0.61,           # 61% hist vol
    risk_free_rate = 4.50,           # 4.50%
    div_yield      = 0.0,
    contract_type  = ContractType.STOCK,
    lot_size       = 100,
)

# add legs from your sheet (negative lots = short)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61, "equity")
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61, "equity")
book.add_option("call", "2026-01-30", 40,   0,  1.80, 0.54, "equity")
book.add_option("call", "2026-06-18", 40,   0,  4.50, 0.61, "equity")
book.add_option("call", "2026-06-18", 50,   0,  7.30, 0.61, "equity")
book.add_option("call", "2026-01-30", 37,   0, 10.40, 0.52, "equity")

# underlying position
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# ── Greeks table ──────────────────────────────────────────────────────────
print(book.greeks_table(days_ahead=0).to_string())

# ── Summary ───────────────────────────────────────────────────────────────
s = book.summary(days_ahead=0)
for k, v in s.items():
    print(f"  {k:20s}: {v}")

# ── +30 days simulation ───────────────────────────────────────────────────
print("\n--- +30 days ---")
s30 = book.summary(days_ahead=30)
for k, v in s30.items():
    print(f"  {k:20s}: {v}")

# ── Save ─────────────────────────────────────────────────────────────────
book.save("positions/INTC.json")
print("\nSaved to positions/INTC.json")

# ── Portfolio ─────────────────────────────────────────────────────────────
port = PortfolioBook(name="My portfolio")
port.add_book(book, path="positions/INTC.json")
print(port.summary())
port.save("positions/portfolio.json")

# ── WatchList ─────────────────────────────────────────────────────────────
wl = WatchList()
wl.add("INTC", notes="short calls position")
wl.add("SPY",  notes="hedge")
wl.save("positions/watchlist.json")

     leg_id         type      expiry strike     iv          model exercise   lots lot_size price_paid model_price      value        pnl     delta  gamma     vega   theta      rho
0  6e527399         Call  2026-12-18   85.0  61.0%  Black-Scholes        E    -60      100      18.11      22.627 -135761.96  -27101.96  -4130.38 -47.00 -1569.90  238.91 -1570.75
1  8768baef         Call  2026-12-18   90.0  61.0%  Black-Scholes        E    -40      100       21.5     20.2986  -81194.30    4805.70  -2582.10 -32.97 -1101.39  165.72 -1005.13
2  a8701431         Call  2026-01-30   40.0  54.0%  Black-Scholes        E      0      100        1.8       53.22       0.00       0.00      0.00   0.00     0.00    0.00     0.00
3  0a64e52d         Call  2026-06-18   40.0  61.0%  Black-Scholes        E      0      100        4.5     53.4513       0.00       0.00      0.00   0.00     0.00   -0.00     0.00
4  ca4a8e54         Call  2026-06-18   50.0  61.0%  Black-Scholes        E      0      100        7.3    

In [52]:
from yfinance_api3.classes.pricing import Space, ContractType
from yfinance_api3.classes.positions_book import PositionsBook, PortfolioBook

# rebuild INTC book
book = PositionsBook(
    symbol="INTC", spot=93.22, vol=0.61,
    risk_free_rate=4.50, div_yield=0.0,)
updates = book.mark_to_market(client, quant)
print(updates)


None


In [50]:





book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61)
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61)
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# single ticker dashboard — replicates your sheet
#plots.positions_book(book, days_ahead=0).show()
#plots.positions_book(book, days_ahead=30).show()   # +30 days simulation

plots.positions_book(book, days_ahead=30).show()   # chart + summary


In [23]:
plots.positions_legs(book, days_ahead=30).show()   # legs table



In [24]:
# portfolio summary
port = PortfolioBook(name="My portfolio")
port.add_book(book)
plots.portfolio_summary(port, days_ahead=0).show()

In [25]:
# Debug GEX
gex = opt.gex_by_strike()
print("GEX sample:")
print(gex.head(5))
print(f"\nMax abs net_gex: {gex['net_gex'].abs().max():.2f}")
print(f"Non-zero rows: {(gex['net_gex'] != 0).sum()} / {len(gex)}")

# Check gamma values
df = opt.greeks(opt.nearest_expiry(0))
print(f"\nGamma stats:")
print(df["gamma"].describe())
print(f"greek_source counts: {df['greek_source'].value_counts().to_dict()}")

GEX sample:
   strike     call_gex      put_gex      net_gex  cumulative_gex
0   400.0  3111.363112 -7915.282729 -4803.919616    -4803.919616
1   405.0  2600.884733  -320.678450  2280.206283    -2523.713334
2   410.0     0.000000 -1645.106522 -1645.106522    -4168.819856
3   415.0    10.428567 -1183.642411 -1173.213843    -5342.033699
4   420.0     0.000000 -4421.712618 -4421.712618    -9763.746317

Max abs net_gex: 725522915.81
Non-zero rows: 213 / 213

Gamma stats:
count    382.000000
mean       0.007388
std        0.023616
min        0.000000
25%        0.000063
50%        0.000452
75%        0.002216
max        0.159276
Name: gamma, dtype: float64
greek_source counts: {'black_scholes': 382}


In [26]:
df = opt.greeks(opt.nearest_expiry(0))
print(df[["type","strike","implied_volatility","gamma","greek_source"]].head(10))
print(f"\nIV stats: min={df['implied_volatility'].min()}, max={df['implied_volatility'].max()}")
print(f"Zero IV rows: {(df['implied_volatility'] == 0).sum()} / {len(df)}")

   type  strike  implied_volatility     gamma   greek_source
0  call   400.0            4.046880  0.000039  black_scholes
1   put   400.0            3.250002  0.000006  black_scholes
2   put   405.0            3.125002  0.000005  black_scholes
3  call   405.0            3.992188  0.000043  black_scholes
4  call   410.0            5.424808  0.000000  black_scholes
5   put   410.0            3.062502  0.000005  black_scholes
6   put   415.0            3.000002  0.000005  black_scholes
7  call   415.0            3.187502  0.000010  black_scholes
8  call   420.0            5.423831  0.000000  black_scholes
9   put   420.0            2.937503  0.000005  black_scholes

IV stats: min=1.0000000000000003e-05, max=5.42480790649414
Zero IV rows: 0 / 382
